In [2]:
# author: Stutie Banerjie
# date: 11.9.2026
# desc: EDA for Data Center Dataset for DTSC 1302 Final Project

### (Preemptive) Research Question
Which socioeconomic characteristics of a census tract (ex. income, poverty, housing costs, education, race, education, etc.) are associated with having a data center? Do the correlations change based on facility size and state?

(Preemptive) Target Variable: 
| Variable | Description | Notes |
| --- | --- | --- |
| has_data_center | Dictates whether or not there is at least one data center in the tract. | 0/1 Flag Variable |

(Preemptive) Predictive Variables
| Variable | Description | Notes |
| --- | --- | --- |
| median_hh_income | Median Household Income in Tract | N/A |
| total_pop | Total Population in Tract | N/A 
| pct_white_nh, pct_black_nh, pct_asian_nh, pct_hispanic, pct_nonwhite | Racial distribution of Tract Population | Would be interesting to see if any, some, or all racial demographies in tract populations had correlation to data centers being build in a specific area. |
| dist_nearest_metro_km | Straight line distance (KM) from tract point to nearest data center | Could potentially show if data centers are built in more rural or urban areas. Might be a grouping variable? |


(Preemptive) Grouping Variables -
| Variable | Description | Options | Notes
| --- | --- | --- | --- |
| states | What state the data was aggregated from. | GA; NC; SC; TN; VA | N/A |
| dc_total_sqft | Total land taken up by data center in city | Range: 1,232 - 26,597,517 | Should probably break these into meaningful categories so building graphs later on is easier.|
| dc_n_campus vs dc_n_building | See what tracts have single building data centers vs multiple buildings for one establishment. | Could be interesting to see if there's a correlation between these categories and different socioeconomic metrics? |

### Clean up dictionary csv file for readability

In [3]:
import pandas as pd

In [4]:
# read in data file
df = pd.read_csv(r'C:\Users\Stutie Banerjie\OneDrive\Documents\Projects\VSCode\dtsc1302-project\0_data\1_data_centers_combined.csv')
df.head()

,geoid,state,county_fips,county_name,total_pop,median_hh_income,poverty_rate,pct_white_nh,pct_black_nh,pct_asian_nh,...,nearest_metro,dist_nearest_metro_km,in_charlotte_region,dc_n_campus,dc_n_building,dc_n_point,dc_n_big_tech,dc_n_operator_unknown,dc_operators,dc_named_facilities
0,13001950100,GA,13001,Appling County,3219,50222.0,19.6,76.7,9.6,0.0,...,Charleston SC,242.1,0,0,0,0,0,0,NaN,NaN
1,13001950201,GA,13001,Appling County,2246,57653.0,21.5,75.6,15.6,1.0,...,Charleston SC,245.1,0,0,0,0,0,0,NaN,NaN
2,13001950202,GA,13001,Appling County,2152,29899.0,24.6,41.2,39.9,0.0,...,Charleston SC,254.4,0,0,0,0,0,0,NaN,NaN
3,13001950301,GA,13001,Appling County,3264,45486.0,25.1,62.3,24.8,0.7,...,Charleston SC,253.2,0,0,0,0,0,0,NaN,NaN
4,13001950302,GA,13001,Appling County,2144,38673.0,29.7,71.1,14.2,1.4,...,Charleston SC,258.4,0,0,0,0,0,0,NaN,NaN


In [5]:
# read in data dictionary
df_dict = pd.read_csv(r'C:\Users\Stutie Banerjie\OneDrive\Documents\Projects\VSCode\dtsc-1302\project_files\1_data_centers\data_dictionary.csv')
df_dict.head()

,file,variable,type,meaning,values_or_range,missing,source
0,1_data_centers_combined.csv,geoid,text,"Census tract ID (11 digits, 2020 tracts). Uniq...","10,599 distinct values, e.g. 13001950100",0,Census
1,1_data_centers_combined.csv,state,text,State abbreviation,GA; NC; SC; TN; VA,0,Census
2,1_data_centers_combined.csv,county_fips,text,County FIPS code (5 digits); keep as text,"533 distinct values, e.g. 13001",0,Census
3,1_data_centers_combined.csv,county_name,text,County name (repeats across states; use with s...,"431 distinct values, e.g. Appling County",0,Census
4,1_data_centers_combined.csv,total_pop,integer,Total population (ACS 5-yr 2020–2024),"2 to 18,754",0,"U.S. Census Bureau, American Community Survey ..."


In [6]:
# filter dictionary to only`include variables that are present in the data
# no duplicates
df_dict_filtered = df_dict.loc[df_dict['variable'].isin(df.columns.tolist())].drop_duplicates(subset='variable')
df_dict_filtered

,file,variable,type,meaning,values_or_range,missing,source
0,1_data_centers_combined.csv,geoid,text,"Census tract ID (11 digits, 2020 tracts). Uniq...","10,599 distinct values, e.g. 13001950100",0,Census
1,1_data_centers_combined.csv,state,text,State abbreviation,GA; NC; SC; TN; VA,0,Census
2,1_data_centers_combined.csv,county_fips,text,County FIPS code (5 digits); keep as text,"533 distinct values, e.g. 13001",0,Census
3,1_data_centers_combined.csv,county_name,text,County name (repeats across states; use with s...,"431 distinct values, e.g. Appling County",0,Census
4,1_data_centers_combined.csv,total_pop,integer,Total population (ACS 5-yr 2020–2024),"2 to 18,754",0,"U.S. Census Bureau, American Community Survey ..."
5,1_data_centers_combined.csv,median_hh_income,integer,"Median household income, $ (ACS 5-yr 2020–2024...","2,499 to 250,001",74,"U.S. Census Bureau, American Community Survey ..."
6,1_data_centers_combined.csv,poverty_rate,decimal,% of people below the poverty line,0 to 100,32,"U.S. Census Bureau, American Community Survey ..."
7,1_data_centers_combined.csv,pct_white_nh,decimal,% non-Hispanic White,0 to 100,0,"U.S. Census Bureau, American Community Survey ..."
8,1_data_centers_combined.csv,pct_black_nh,decimal,% non-Hispanic Black,0 to 100,0,"U.S. Census Bureau, American Community Survey ..."
9,1_data_centers_combined.csv,pct_asian_nh,decimal,% non-Hispanic Asian,0 to 67.9,0,"U.S. Census Bureau, American Community Survey ..."


In [7]:
# check
print("dictionary rows:", len(df_dict))
print("data columns:", len(df.columns))
print("filtered rows:", len(df_dict_filtered))

dictionary rows: 91
data columns: 39
filtered rows: 39


In [8]:
# save this dictionary 
# df_dict_filtered.to_csv("data_dictionary_filtered.csv", index=False)

### Data Description

In [9]:
# check rows & columns

df.shape

# 10,599 rows, 39 columns 


(10599, 39)

### Data Cleaning

In [13]:
rows = []

# iterate and add to list 
for col in df.columns:
    null_num = df[col].isna().sum()
    rows.append({
        "column": col,
        "missing": null_num,
        "pct_missing": round(null_num / len(df) * 100, 2)
    })

# show columns w/ highest percentage of missing values first
missing_values = pd.DataFrame(rows)
missing_values = missing_values.sort_values("pct_missing", ascending=False)
print(missing_values)

# step 1:variables >= 95% missing values
    # dc_operators, dc_largest_sqft, dc_named facilities
# these variables might be more worth it to just drop 

# step 2: variables with minimal missing values
    # median_gross_rent, median_home_value, median_hh_income,
    # pct_rent_burdened, pct_owner_occupied, pct_no_vehicle, 
    # poverty_rate, unemployment_rate, pct_bachelors_plus
# i think we should just drop rows where these variables are missing for data normalcy

                   column  missing  pct_missing
37           dc_operators    10540        99.44
38    dc_named_facilities    10518        99.24
25        dc_largest_sqft    10518        99.24
17      median_gross_rent      546         5.15
18      median_home_value      229         2.16
5        median_hh_income       74         0.70
16      pct_rent_burdened       50         0.47
14     pct_owner_occupied       37         0.35
15         pct_no_vehicle       37         0.35
6            poverty_rate       32         0.30
13      unemployment_rate       22         0.21
12     pct_bachelors_plus        1         0.01
24          dc_total_sqft        0         0.00
35          dc_n_big_tech        0         0.00
30  dist_nearest_metro_km        0         0.00
31    in_charlotte_region        0         0.00
32            dc_n_campus        0         0.00
33          dc_n_building        0         0.00
34             dc_n_point        0         0.00
29          nearest_metro        0      

In [14]:
# duplicate rows = 0
df.duplicated().sum()

# duplicate geoid values = 0
df['geoid'].duplicated().sum()

np.int64(0)

In [ ]:
# incorrect entries

# check that all populations are at least 1 
print(f"Total Populations less than 1: {(df['total_pop'] < 1).sum()}")

# check that land_sqmi is positive for each tract
print(f"Total Land Areas less than 0: {(df['land_sqmi'] < 0).sum()}")

# check that only 5 states are represented
print(f"Number of states represented: {df['state'].nunique()}")


Total Populations less than 1: 0
Total Land Areas less than 0: 0
Number of states represented: 5


In [25]:
# count # of placeholder values 
print(f"Household Income Cap Count: {(df['median_hh_income'] == 250001).sum()}")
print(f"Gross Rent Cap Count: {(df['median_gross_rent'] == 3501).sum()}")
print(f"Home Value Cap Count: {(df['median_home_value'] == 2000001).sum()}")

Household Income Cap Count: 75
Gross Rent Cap Count: 85
Home Value Cap Count: 4


In [ ]:
# internal consistency checks

# check 1: racial percentages add to 1
bad_rows = []

# iterate through df row at a time
# i is row index, r is row itself
for i, r in df.iterrows():
    # 0.5 = tolerance for rounding errors
    if abs(r['pct_nonwhite'] - (100 - r['pct_white_nh'])) > 0.5:
        bad_rows.append(i)

print("Rows with errors:", len(bad_rows))
print("Row indexes:", bad_rows)


Rows with errors: 0
Row indexes: []


In [ ]:
# check 2: data center information accuracy
wrong_count = []

for i, r in df.iterrows():
    # has_data_center should be 1 if n_data_centers > 0
    if (r['has_data_center'] == 1) != (r['n_data_centers'] > 0):
        wrong_count.append(i)
    # dc_total_sqft should be 0 if has_data_center is 0
    if r['has_data_center'] == 0 and r['dc_total_sqft'] != 0:
        wrong_count.append(i)
    # dc_largest_sqft <= dc_total_sqft
    if r['dc_largest_sqft'] > r['dc_total_sqft']:
        wrong_count.append(i)
    # pop_density_sqmi = total_pop / land_sqmi
    # 0.5 = tolerance for rounding errors
    if abs(r['pop_density_sqmi'] - r['total_pop'] / r['land_sqmi']) > 0.5:
        wrong_count.append(i)

print("Rows with errors:", len(wrong_count))
print("Row indexes:", wrong_count)

Rows with errors: 0
Row indexes: []


In [41]:
# check 3: geoid accuracy aka county_fips should be first 5 digits of geoid
bad_geoid_rows = []

for i, r in df.iterrows():
    # zfill keeps leading 0's
    fips_id = str(r['county_fips']).zfill(5)
    # gets first 5 digits of geoid
    geo_id = str(r['geoid']).zfill(11)[:5]
    if fips_id != geo_id:
        bad_geoid_rows.append(i)

print("Rows with errors:", len(bad_geoid_rows))
print("Row indexes:", bad_geoid_rows)

Rows with errors: 0
Row indexes: []


In [ ]:
# delete columns based on step 1
df = df.drop(columns=['dc_operators', 'dc_largest_sqft', 'dc_named_facilities'])

# reasoning: mostly null columns, not useful for analysis

df

,geoid,state,county_fips,county_name,total_pop,median_hh_income,poverty_rate,pct_white_nh,pct_black_nh,pct_asian_nh,...,dist_nearest_dc_km,n_dc_within_10km,nearest_metro,dist_nearest_metro_km,in_charlotte_region,dc_n_campus,dc_n_building,dc_n_point,dc_n_big_tech,dc_n_operator_unknown
0,13001950100,GA,13001,Appling County,3219,50222.0,19.6,76.7,9.6,0.0,...,200.29,0,Charleston SC,242.1,0,0,0,0,0,0
1,13001950201,GA,13001,Appling County,2246,57653.0,21.5,75.6,15.6,1.0,...,207.52,0,Charleston SC,245.1,0,0,0,0,0,0
2,13001950202,GA,13001,Appling County,2152,29899.0,24.6,41.2,39.9,0.0,...,210.28,0,Charleston SC,254.4,0,0,0,0,0,0
3,13001950301,GA,13001,Appling County,3264,45486.0,25.1,62.3,24.8,0.7,...,216.37,0,Charleston SC,253.2,0,0,0,0,0,0
4,13001950302,GA,13001,Appling County,2144,38673.0,29.7,71.1,14.2,1.4,...,215.21,0,Charleston SC,258.4,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10594,51840000102,VA,51840,Winchester city,3551,39808.0,43.6,55.6,11.3,1.2,...,22.07,0,Washington DC,101.7,0,0,0,0,0,0
10595,51840000201,VA,51840,Winchester city,3524,67206.0,18.7,70.1,6.0,6.4,...,22.31,0,Washington DC,103.3,0,0,0,0,0,0
10596,51840000202,VA,51840,Winchester city,5771,78750.0,15.3,70.8,3.0,4.1,...,20.68,0,Washington DC,103.6,0,0,0,0,0,0
10597,51840000301,VA,51840,Winchester city,5346,66821.0,11.2,58.4,9.2,0.3,...,20.51,0,Washington DC,101.7,0,0,0,0,0,0
